# 01. 문서 요약 Tool — PDF 전처리 + `summary.py`
**Day 4**

> 수업용 문서: **`samples/Language_Models.pdf`**
> (언어 모델 개요 자료 — 긴 PDF 요약 실습에 적합)

**파이프라인:** `PDF → PyMuPDF 추출 → 전처리 → OpenAI 요약 → 저장`

---
## 0. 설치

In [1]:
!pip install pymupdf openai python-dotenv

In [ ]:
!pip install pymupdf #pymupdf : pdf 읽을 수 있는 library

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv() #API key 불러오기

True

In [4]:
pdf_path = os.path.join(os.getcwd(),"samples/Language_models.pdf")

---
## 1. PyMuPDF로 `Language_Models.pdf` 텍스트 추출

In [5]:
import pymupdf

doc = pymupdf.open(pdf_path)

In [ ]:
doc #generator 역할이 있음

Document('c:\Users\Admin\OneDrive\바탕 화면\실습\4일차\samples/Language_models.pdf')

In [7]:
i=0
for page in doc:
    text = page.get_text()
    print(text)
    i+=1
    if i==5:
        break
#각 페이지 별로 get.text 진행 -> text 출력 

Language Models are Few-Shot Learners
Tom B. Brown∗
Benjamin Mann∗
Nick Ryder∗
Melanie Subbiah∗
Jared Kaplan†
Prafulla Dhariwal
Arvind Neelakantan
Pranav Shyam
Girish Sastry
Amanda Askell
Sandhini Agarwal
Ariel Herbert-Voss
Gretchen Krueger
Tom Henighan
Rewon Child
Aditya Ramesh
Daniel M. Ziegler
Jeffrey Wu
Clemens Winter
Christopher Hesse
Mark Chen
Eric Sigler
Mateusz Litwin
Scott Gray
Benjamin Chess
Jack Clark
Christopher Berner
Sam McCandlish
Alec Radford
Ilya Sutskever
Dario Amodei
OpenAI
Abstract
Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training
on a large corpus of text followed by ﬁne-tuning on a speciﬁc task. While typically task-agnostic
in architecture, this method still requires task-speciﬁc ﬁne-tuning datasets of thousands or tens of
thousands of examples. By contrast, humans can generally perform a new language task from only
a few examples or from simple instructions – something which current NLP systems still largely
struggle

In [9]:
full_text=''
for page in doc:
    text = page.get_text()
    full_text+=text + '\n------------------------\n'
#full_text에 page별로 text 저장 -> text 넘어가면 ---------------------로 구분

len(full_text)

238620

**Txt 파일로 저장하는 법**

In [11]:
os.path.basename(pdf_path)

'Language_models.pdf'

In [10]:
pdf_file_name = os.path.basename(pdf_path)

In [ ]:
pdf_file_name.split('.')[0]+'.txt' #txt 확장자로 저장

'Language_models.txt'

In [19]:
pdf_file_name = os.path.splitext(pdf_file_name)[0] #'.' 기준 파일명 추출
txt_file_path = os.path.join(os.getcwd(),'samples',f"{pdf_file_name}.txt") #파일 확장자 txt 변경 및 path 설정
txt_file_path

'c:\\Users\\Admin\\OneDrive\\바탕 화면\\실습\\4일차\\samples\\Language_models.txt'

In [20]:
with open(txt_file_path, 'w', encoding='utf-8') as f:
    f.write(full_text) #pdf의 언어를 full_text.txt로 저장

---
## 2. 문서 요약

In [22]:
api_key = os.getenv('OPENAI_API_KEY')

def summarize_txt(file_path): #file_path : txt
    client = OpenAI(api_key=api_key)
    with open(file_path, 'r', encoding = 'utf-8') as f:
        txt = f.read()

    system_prompt = f'''
    너는 다음 글을 요약하는 봇이다. 아래 글을 읽고, 

    작성해야 하는 포맷은 다음과 같음
    # 제목

    ## 저자의 문제 인식 및 주장 (15문장 이내)

    ## 저자 소개


    ============= 이하 텍스트 ================
    {txt[:10000]}

    '''

    response = client.chat.completions.create(
        model = 'gpt-4o-mini',
        temperature = 0.1,
        messages=[
            {"role":"system","content":system_prompt},
        ]
    )

    return response.choices[0].message.content


In [25]:
summary = summarize_txt(txt_file_path)

In [26]:
print(summary)

# GPT-3: Few-Shot Learning in Language Models

## 저자의 문제 인식 및 주장
저자들은 최근 자연어 처리(NLP) 분야에서 대규모 텍스트 코퍼스를 기반으로 한 사전 학습과 특정 작업에 대한 미세 조정이 많은 성과를 거두었다고 주장한다. 그러나 이러한 방법은 여전히 수천 또는 수만 개의 작업별 데이터셋을 필요로 하며, 이는 새로운 언어 작업을 수행하는 데 있어 인간이 몇 가지 예시나 간단한 지시만으로도 가능하다는 점에서 한계를 가진다. 저자들은 언어 모델의 규모를 확장함으로써 이러한 작업 비특이적, 소수 샷 성능이 크게 향상될 수 있음을 보여준다. 특히, 1750억 개의 매개변수를 가진 GPT-3를 훈련시켜, 미세 조정 없이도 다양한 NLP 데이터셋에서 강력한 성능을 발휘할 수 있음을 입증하였다. GPT-3는 번역, 질문-응답, 클로즈 작업 등에서 우수한 성과를 보였으며, 즉석에서 추론이나 도메인 적응이 필요한 작업에서도 효과적이었다. 그러나 여전히 몇몇 데이터셋에서는 성능이 저조하거나 훈련 과정에서 발생하는 방법론적 문제를 발견하였다. 마지막으로, GPT-3는 인간이 작성한 기사와 구별하기 어려운 뉴스 기사를 생성할 수 있으며, 이러한 발견의 사회적 영향에 대해서도 논의하였다.

## 저자 소개
저자들은 OpenAI 소속의 연구자들로, Tom B. Brown, Benjamin Mann, Nick Ryder, Melanie Subbiah, Jared Kaplan 등 다양한 전문가들이 포함되어 있다. 이들은 자연어 처리 및 인공지능 분야에서의 연구와 개발에 기여하고 있으며, GPT-3와 같은 혁신적인 언어 모델의 개발에 중요한 역할을 하고 있다.


## 실습 : 함수 생성

- input : pdf 파일 경로 
- output : pdf 내용을 요약한 summary.txt 파일 생성 및 summary return

- 파일명: pdf_summary.py

In [28]:
from summary_answer import summarize_pdf
summary = summarize_pdf("samples/학칙.pdf")
print(summary)

# 대학원 학칙 요약

## 저자의 문제 인식 및 주장
대학원 학칙은 기독교 정신을 바탕으로 창의적 이론과 과학적 방법을 탐구하고 인류 문화 향상에 기여하는 것을 목적으로 한다. 학칙은 석사 및 박사학위 과정, 통합과정, 협동과정 등 다양한 학위과정을 규정하고 있으며, 각 과정의 수업연한과 재학연한을 명시하고 있다. 또한, 학생의 입학 및 전형 방법, 등록 및 학적 변동, 수업 및 학점 인정, 자격시험 및 수료 요건 등을 상세히 규정하고 있다. 특히, 학위논문 제출 자격시험과 수료 요건은 학위 취득의 중요한 기준으로 설정되어 있다. 학칙은 학생의 권리와 의무를 명확히 하여 학업의 질을 높이고, 학위 취득 과정에서의 공정성을 보장하고자 한다. 또한, 학생의 자치활동과 징계에 관한 규정도 포함되어 있어 학생의 자율성과 책임을 강조하고 있다. 이러한 규정들은 대학원 교육의 체계성과 질적 향상을 도모하기 위한 것이다.

## 저자 소개
저자는 대학원 학칙을 제정하고 개정하는 과정에 참여한 교육 관계자들로, 대학원 교육의 질적 향상과 학생들의 학습 환경 개선을 위해 노력하고 있다. 이들은 기독교 정신을 바탕으로 한 교육 철학을 지니고 있으며, 학생들이 창의적이고 과학적인 사고를 통해 인류 문화에 기여할 수 있도록 지원하고 있다.


In [35]:
from test_summary import summarize_pdf_test
summary = summarize_pdf_test("samples/Language_Models.pdf")
print(summary)

# GPT-3: Few-Shot Learning in Language Models

## 저자의 문제 인식 및 주장
저자들은 최근 자연어 처리(NLP) 분야에서 대규모 텍스트 코퍼스를 기반으로 한 사전 훈련과 특정 작업에 대한 미세 조정이 많은 성과를 거두었다고 주장한다. 그러나 이러한 방법은 여전히 수천 또는 수만 개의 작업별 데이터셋을 필요로 하며, 이는 새로운 언어 작업을 수행하는 데 있어 인간이 몇 가지 예시나 간단한 지시만으로도 가능하다는 점에서 한계를 가진다. 저자들은 언어 모델의 규모를 확장하면 작업에 구애받지 않는 몇 가지 샷 성능이 크게 향상된다고 주장하며, GPT-3라는 1750억 개의 매개변수를 가진 자가 회귀 언어 모델을 훈련하여 이를 입증한다. GPT-3는 미세 조정 없이도 다양한 NLP 데이터셋에서 강력한 성능을 발휘하며, 번역, 질문 응답, 클로즈 작업 등에서 경쟁력을 보인다. 그러나 저자들은 GPT-3가 여전히 몇몇 데이터셋에서 어려움을 겪고 있으며, 대규모 웹 코퍼스에서 훈련된 것과 관련된 방법론적 문제도 존재한다고 지적한다. 또한, GPT-3가 생성한 뉴스 기사가 인간 작성자와 구별하기 어려운 수준에 이르렀음을 강조하며, 이러한 발견이 사회에 미치는 광범위한 영향을 논의한다.

## 저자 소개
저자들은 OpenAI 소속의 연구자들로, Tom B. Brown, Benjamin Mann, Nick Ryder, Melanie Subbiah, Jared Kaplan 등 다양한 전문가들이 포함되어 있다. 이들은 자연어 처리 및 기계 학습 분야에서의 경험을 바탕으로 GPT-3의 개발과 연구에 기여하였다.
